# Обучение YOLOv8 — Fashionpedia (11 классов)

Baseline и модель 1 из 5. Для **честного сравнения архитектур** все 5 моделей обучаются на **одинаковой подвыборке train ~3000 изображений** (те же файлы, что у Faster R-CNN/SSD/EfficientDet/DETR) и оцениваются на **полном test** (4437).

Ноутбук рассчитан на Kaggle с **GPU** и **Internet**.

## 0. Установка и проверка Ultralytics

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import ultralytics, torch
ultralytics.checks()
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Клонирование репозитория с кодом

При повторном запуске `git clone` падает («destination path already exists») — поэтому pull при наличии папки.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/artemdev2281/cv-fashion-object-detection.git"  
REPO_DIR = "/kaggle/working/cv-fashion-object-detection"

if os.path.exists(REPO_DIR):
    print("Репозиторий уже склонирован — обновляю (git pull)")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("sys.path[0]:", sys.path[0])

## 2. Починка `data.yaml`

Данные подключены read-only в `/kaggle/input/...`, а `path:` в `data.yaml` устарел. Копируем в `/kaggle/working/` и подменяем `path:`.

In [ ]:
import yaml
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/notebooks/neuromindgpt/"
    "cv-fashion-object-detection/cv-fashion-object-detection/data/processed"
)
assert DATA_ROOT.exists(), f"Не найден каталог данных: {DATA_ROOT}. Проверьте Add Input."

src_yaml = DATA_ROOT / "data.yaml"
data = yaml.safe_load(open(src_yaml, encoding="utf-8"))
data["path"] = str(DATA_ROOT)

WORK_YAML = Path("/kaggle/working/data.yaml")
yaml.safe_dump(data, open(WORK_YAML, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

print("nc:", data["nc"], "| names:", data["names"])
for split in ("train", "val", "test"):
    d = DATA_ROOT / data[split]
    n = len(list(d.glob("*.jpg"))) if d.exists() else 0
    print(f"{split:5} -> {'OK' if d.exists() else 'НЕ НАЙДЕНО'}, {n} изображений")

## 3. Подвыборка train 3000 (те же файлы, что у остальных моделей)

Берём первые 3000 изображений по отсортированному `image_id` из `coco/train.json` — ровно те же, что использует `CocoDetectionDataset` для Faster R-CNN/SSD. Формируем список путей и `data.yaml` подвыборки: **train = 3000, val/test = полные**.

In [ ]:
import json

train_coco = json.load(open(DATA_ROOT / "coco" / "train.json", encoding="utf-8"))
ids = sorted(img["id"] for img in train_coco["images"])[:3000]
id_to_file = {img["id"]: img["file_name"] for img in train_coco["images"]}

subset_txt = Path("/kaggle/working/train_subset_3000.txt")
with open(subset_txt, "w", encoding="utf-8") as f:
    for i in ids:
        f.write(str(DATA_ROOT / id_to_file[i]) + "\n")
print("Подвыборка train:", len(ids), "изображений ->", subset_txt)

data_sub = dict(data)
data_sub["train"] = str(subset_txt)   # абсолютный путь к списку изображений
SUBSET_YAML = Path("/kaggle/working/data_subset3000.yaml")
yaml.safe_dump(data_sub, open(SUBSET_YAML, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print("data.yaml подвыборки:", SUBSET_YAML)

## 4. Обучение (20 эпох, подвыборка 3000, оценка на полном test)

In [ ]:
from src.utils.utils import load_config
from src.models.yolo import build_model
from src.training.train import train

config = load_config(Path(REPO_DIR) / "configs" / "default.yaml")
num_classes = data["nc"]

model = build_model(num_classes, config=config, data_yaml=str(SUBSET_YAML))

metrics = train(
    model,
    str(SUBSET_YAML),
    config,
    epochs=20,
    split="test",              # честная оценка на полном отложенном test
    project="/kaggle/working/results/logs",
    name="yolov8_subset3000",
)

## 5. Итоговые метрики (общие + per-class)

In [ ]:
import pandas as pd

print("=== YOLOv8 @ train3000 (test) ===")
print(f"mAP@0.5:      {metrics['map50']:.4f}")
print(f"mAP@0.5:0.95: {metrics['map50_95']:.4f}")
print(f"Precision:    {metrics['precision']:.4f}")
print(f"Recall:       {metrics['recall']:.4f}")
print(f"F1:           {metrics['f1']:.4f}")
print(f"Изображений в test: {metrics['num_images']}")

df = (
    pd.DataFrame(metrics["per_class"]).T[["map50", "map50_95", "precision", "recall", "f1"]]
    .sort_values("map50_95", ascending=False)
)
df.round(4)

In [ ]:
best = df.index[0]; worst = df.index[-1]
print(f"Лучший класс:  {best}  (mAP@0.5:0.95 = {df.loc[best, 'map50_95']:.4f})")
print(f"Худший класс:  {worst}  (mAP@0.5:0.95 = {df.loc[worst, 'map50_95']:.4f})")

## 6. Сохранение результатов + Save Version

Веса `best.pt`, метрики и графики Ultralytics лежат в `/kaggle/working/results/logs/yolov8_subset3000/`. Собираем сводку и резервный zip; в конце — **Save Version**.

In [ ]:
import json, shutil

OUT = Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

json.dump(metrics, open(OUT / "metrics_yolov8_subset3000.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
df.to_csv(OUT / "yolov8_subset3000_per_class.csv", encoding="utf-8")

shutil.make_archive("/kaggle/working/yolov8_subset3000_results", "zip", "/kaggle/working/results")

train_dir = Path(metrics["save_dir"]).parent / "yolov8_subset3000"
print("Веса:", list((train_dir / "weights").glob("*.pt")))
print("Графики:", [p.name for p in train_dir.glob("*.png")])
print()
print("Скачайте /kaggle/working/yolov8_subset3000_results.zip из Output и/или Save Version.")